# Classical Text Classification -- SOLUTION

---

<img src="https://www.dropbox.com/scl/fi/b1vbv4c4m5vikt6s08n62/nlp.png?rlkey=r5t9i1socnr84jk2slvx2pylw&raw=1" align="center"/>

## The Scenario

You are a data scientist. You just inherited a folder of **10,000 Yelp reviews** with star ratings. Your manager wants to know, by tomorrow, whether a given review is **positive** (4-5 stars) or **negative** (1-2 stars).

You have:
- No GPU.
- No labeled "sentiment" column (only star ratings).
- One afternoon.
- A laptop with `scikit-learn` installed.

Deep learning is overkill. What you need is a **strong classical baseline** that a future neural network will have to beat. That is what we will build today.

## Learning Objectives

By the end of this notebook you will be able to:
1. Frame a text problem as a supervised learning task.
2. Convert raw text into numeric features with `CountVectorizer` (Bag of Words) and `TfidfVectorizer` (TF-IDF).
3. Use n-grams and stop words, and know when each helps.
4. Train and evaluate **Multinomial Naive Bayes**, **Random Forest**, and **Logistic Regression** on text.
5. Read a `classification_report` critically (precision, recall, F1) and compare models with confusion matrices.
6. Engineer hand-crafted features (review length, punctuation, sentiment) and combine them with a TF-IDF matrix via `scipy.sparse.hstack`.
7. Apply the full pipeline to a brand-new dataset (SMS spam) to prove the workflow generalizes.

## Prerequisites
- Notebook 1 (preprocessing basics: tokenization, stop words, stemming).
- Comfort with `pandas` and `numpy`.

## The Golden Rule of Classical ML for Text

> **Always compute the majority-class baseline first.** If 82% of your reviews are 5-star, a model that always predicts "5 stars" gets 82% accuracy. Any real model must beat that bar. We will enforce this discipline throughout.



## Section 0: Environment Setup

Run the next cell once per Colab session. If you are on a local machine with `requirements.txt` already installed, you can skip it.

In [ ]:
# Install the small set of libraries we need for classical text classification.
!pip install --quiet --upgrade scikit-learn textblob spacy pandas numpy matplotlib seaborn scipy

### The Classifier Pipeline

Every classical text classifier we will build today follows the same three-step pipeline:

```
  raw text  ->  vectorizer  ->  classifier  ->  prediction
```

The vectorizer turns strings into a sparse numeric matrix (rows = documents, columns = words). The classifier is a standard scikit-learn estimator that knows nothing special about text.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.sparse as sp

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

from textblob import TextBlob

SEED            = 42
TEST_SIZE       = 0.2
MAX_FEATURES    = 20000
NGRAM_RANGE     = (1, 2)
MIN_DF          = 2
MAX_DF          = 0.95
RF_N_ESTIMATORS = 100

np.random.seed(SEED)
sns.set_theme(style='whitegrid')
print('Environment setup complete.')

## Section 1: Load Yelp & Frame the Problem

Binary filter: keep 1-star and 5-star reviews only, label `1` for 5-star (positive) and `0` for 1-star (negative).

In [ ]:
YELP_URL = 'https://www.dropbox.com/s/xds4lua69b7okw8/yelp.csv?dl=1'
yelp = pd.read_csv(YELP_URL)

print(f'Full dataset: {len(yelp):,} rows')
print(yelp['stars'].value_counts().sort_index())

yelp_bw = yelp[yelp['stars'].isin([1, 5])].reset_index(drop=True)
print(f'\nAfter 1/5-star filter: {len(yelp_bw):,} rows')

X = yelp_bw['text']
y = (yelp_bw['stars'] >= 4).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)

print(f'\nTrain: {len(X_train):,}   Test: {len(X_test):,}')
print(f'Positive rate (train): {y_train.mean():.3f}')

## Section 2: Bag of Words

Count how often each word appears in each document. The resulting matrix has one row per document and one column per unique word. Order is lost -- hence "bag of words".

In [ ]:
toy = ["I love pizza", "pizza is great", "I hate pizza"]
demo_vect = CountVectorizer()
demo_dtm  = demo_vect.fit_transform(toy)

print('Vocabulary:', demo_vect.vocabulary_)
print('\nDocument-term matrix:')
print(pd.DataFrame(demo_dtm.toarray(),
                   columns=demo_vect.get_feature_names_out(),
                   index=[f'doc {i}' for i in range(3)]))

In [ ]:
bow = CountVectorizer()
X_train_bow = bow.fit_transform(X_train)
X_test_bow  = bow.transform(X_test)

print(f'X_train_bow shape: {X_train_bow.shape}')
print(f'Vocabulary size:   {len(bow.vocabulary_):,}')
print(f'Non-zero entries:  {X_train_bow.nnz:,}')
print(f'Density:           {X_train_bow.nnz / np.prod(X_train_bow.shape):.4%}')

### Sparse Matrices & the Majority-Class Baseline

Density ~0.5%. Storing dense would waste hundreds of MB. scikit-learn uses `scipy.sparse.csr_matrix` -- all our classifiers accept sparse input. Only call `.toarray()` on small slices for inspection.

Before training anything, compute the number any real model must beat.

In [ ]:
majority_class = y_train.mode()[0]
baseline_preds = np.full(len(y_test), majority_class)
baseline_acc   = accuracy_score(y_test, baseline_preds)

print(f'Majority class in train: {majority_class}  (1 = positive)')
print(f'Baseline accuracy:       {baseline_acc:.4f}')

### Lab 2.1 -- Vocabulary Size vs. Accuracy

Test three `max_features` values and plot downstream NB accuracy.

In [ ]:
# Solution: Lab 2.1
feature_sizes = [1000, 5000, 20000]
accuracies    = []

for n in feature_sizes:
    # 1. Build vectorizer capped at n features (most frequent terms kept)
    vect = CountVectorizer(max_features=n)
    # 2. Fit on train, transform both sets
    Xtr = vect.fit_transform(X_train)
    Xte = vect.transform(X_test)
    # 3. Train NB
    clf = MultinomialNB().fit(Xtr, y_train)
    # 4. Score
    acc = accuracy_score(y_test, clf.predict(Xte))
    accuracies.append(acc)
    print(f'max_features={n:>6}   accuracy={acc:.4f}')

# 5. Plot
plt.figure(figsize=(7, 4))
plt.semilogx(feature_sizes, accuracies, marker='o')
plt.xlabel('max_features')
plt.ylabel('test accuracy')
plt.title('Vocabulary size vs. Naive Bayes accuracy')
plt.grid(True, which='both', alpha=0.3)
plt.show()

# Explanation:
# Accuracy typically rises from 1000 -> 5000 (enough vocabulary to capture
# sentiment-bearing words) then plateaus. Going from 5000 -> 20000 adds mostly
# rare/noisy tokens that do not shift NB predictions much. Lesson: bigger
# vocabulary is not always better -- there is a sweet spot, and it pays to find it.

## Section 3: Stop Words & N-grams

Stop words = common words like `the`, `is`, `a`. Dropping them shrinks the vocab and usually reduces noise -- **except** in sentiment tasks where `not` is a stop word and removing it flips meaning.

N-grams: unigrams are single words, bigrams are two consecutive words. `ngram_range=(1, 2)` captures both.

In [ ]:
configs = {
    'unigrams, no stop words':  CountVectorizer(ngram_range=(1, 1)),
    'unigrams + english stops': CountVectorizer(ngram_range=(1, 1), stop_words='english'),
    'unigrams + bigrams':       CountVectorizer(ngram_range=(1, 2), min_df=MIN_DF),
}

for name, v in configs.items():
    Xtr = v.fit_transform(X_train)
    Xte = v.transform(X_test)
    nb  = MultinomialNB().fit(Xtr, y_train)
    acc = accuracy_score(y_test, nb.predict(Xte))
    print(f'{name:30}  vocab={Xtr.shape[1]:>6}   acc={acc:.4f}')

In [ ]:
v = CountVectorizer(ngram_range=(2, 2), min_df=5)
v.fit(X_train)
bigrams = [b for b in v.get_feature_names_out()
           if 'not' in b.split() or 'very' in b.split()]
print('A few negation/intensity bigrams:')
print(bigrams[:15])

### Lab 3.1 -- Custom Domain Stop Words

Extend sklearn's English list with restaurant-specific fillers and measure the effect.

In [ ]:
# Solution: Lab 3.1
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

extra_stops = ['restaurant', 'food', 'place', 'time', 'order']

# 1. Combine sklearn's list with our domain additions
my_stops = list(ENGLISH_STOP_WORDS) + extra_stops

# 2. Vectorize with the combined list + bigrams
vect_custom = CountVectorizer(stop_words=my_stops, ngram_range=(1, 2), min_df=MIN_DF)
Xtr = vect_custom.fit_transform(X_train)
Xte = vect_custom.transform(X_test)

# 3. Train + score
nb_custom = MultinomialNB().fit(Xtr, y_train)
acc_custom = accuracy_score(y_test, nb_custom.predict(Xte))
print(f'Custom domain stop words accuracy: {acc_custom:.4f}')

# Explanation:
# Removing high-frequency domain-neutral words (`restaurant`, `food`, `place`) should
# leave accuracy roughly unchanged -- they appear in both positive and negative reviews
# equally, so they carry no sentiment signal. The benefit is a slimmer vocabulary
# (faster inference, smaller models). A common mistake is to toss in sentiment-bearing
# words like `good`, `bad`, `not` -- that *would* hurt accuracy dramatically.

## Section 4: TF-IDF

`tfidf(t, d) = tf(t, d) * idf(t)` where `idf(t) = log(N / df(t)) + 1`.
Rare words get high IDF. Words in every document get near-zero IDF.

In [ ]:
tfidf_demo = TfidfVectorizer(min_df=MIN_DF, max_df=MAX_DF, max_features=MAX_FEATURES)
tfidf_demo.fit(X_train)

idf_series = pd.Series(tfidf_demo.idf_, index=tfidf_demo.get_feature_names_out())
print('Lowest IDF (near-useless):')
print(idf_series.nsmallest(10).round(3))
print('\nHighest IDF (rare, informative):')
print(idf_series.nlargest(10).round(3))

In [ ]:
tfidf = TfidfVectorizer(
    ngram_range=NGRAM_RANGE, min_df=MIN_DF, max_df=MAX_DF,
    max_features=MAX_FEATURES, stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

nb_tfidf = MultinomialNB().fit(X_train_tfidf, y_train)
pred_tfidf = nb_tfidf.predict(X_test_tfidf)
print(f'NB + TF-IDF accuracy: {accuracy_score(y_test, pred_tfidf):.4f}')
print(f'Vocabulary size:      {X_train_tfidf.shape[1]:,}')

### Lab 4.1 -- Compute TF-IDF by Hand

Reproduce sklearn's output on a 3-document toy corpus.

In [ ]:
# Solution: Lab 4.1
simple_train = ['call you tonight', 'call me a cab', 'please call me please']

# 1. Vocabulary (sklearn drops single-char tokens like 'a' by default)
vocab = sorted(['call', 'you', 'tonight', 'me', 'cab', 'please'])
V = len(vocab)
vocab_idx = {w: i for i, w in enumerate(vocab)}

# 2. Term-frequency matrix
tf = np.zeros((3, V), dtype=float)
for i, doc in enumerate(simple_train):
    for w in doc.split():
        if w in vocab_idx:       # skips single-char 'a'
            tf[i, vocab_idx[w]] += 1

# 3. Document frequency (# docs each term appears in)
df = (tf > 0).sum(axis=0)

# 4. IDF -- sklearn uses the smoothed formula by default, but the classic
#    log(N/df) + 1 version matches when smooth_idf=False in the demo.
N = 3
idf = np.log(N / df) + 1

# 5. tf-idf then L2-normalize each row (sklearn default norm='l2')
tfidf_raw = tf * idf
row_norms = np.linalg.norm(tfidf_raw, axis=1, keepdims=True)
row_norms[row_norms == 0] = 1  # avoid division by zero
tfidf_manual = tfidf_raw / row_norms

# 6. Compare -- we disable smoothing to match our manual formula
tfidf_sklearn = TfidfVectorizer(smooth_idf=False).fit_transform(simple_train).toarray()

print('Manual:')
print(np.round(tfidf_manual, 3))
print('\nsklearn (smooth_idf=False):')
print(np.round(tfidf_sklearn, 3))
print('\nMatch?', np.allclose(tfidf_manual, tfidf_sklearn, atol=1e-3))

# Explanation:
# The ingredients are simple: tf (counts), idf (log-inverse-document-frequency),
# multiply, L2-normalize. The only two gotchas: (a) sklearn uses smooth_idf=True
# by default which adds 1 to both numerator and denominator to avoid log(0) for
# unseen terms, and (b) sklearn normalizes each row to unit L2 norm so row vectors
# can be compared with cosine similarity directly.

## Section 5: Classifiers -- NB, RF, Logistic Regression

MNB is the canonical text baseline: fast, robust, handles sparse high-dim input.
RF captures non-linear interactions but is memory-hungry on sparse text.
LR is the industry workhorse with calibrated probabilities.

In [ ]:
models = {
    'MultinomialNB'      : MultinomialNB(),
    'RandomForest'       : RandomForestClassifier(n_estimators=RF_N_ESTIMATORS, random_state=SEED, n_jobs=-1),
    'LogisticRegression' : LogisticRegression(max_iter=1000, random_state=SEED),
}

predictions = {}
for name, clf in models.items():
    clf.fit(X_train_tfidf, y_train)
    preds = clf.predict(X_test_tfidf)
    predictions[name] = preds
    print(f'{name:22} accuracy = {accuracy_score(y_test, preds):.4f}')

print(f'\nMajority-class baseline = {baseline_acc:.4f}')

In [ ]:
for name, preds in predictions.items():
    print(f'=== {name} ===')
    print(classification_report(y_test, preds, target_names=['negative (1*)', 'positive (5*)']))

### Reading the `classification_report`

- **precision = TP / (TP + FP)** -- of what I flagged positive, how many were truly?
- **recall    = TP / (TP + FN)** -- of the actual positives, how many did I catch?
- **f1**       -- harmonic mean.

Spam -> precision matters. Cancer screening -> recall matters. Balanced -> F1.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, preds) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['neg', 'pos'], yticklabels=['neg', 'pos'], ax=ax, cbar=False)
    ax.set_title(f'{name}\nacc={accuracy_score(y_test, preds):.3f}')
    ax.set_xlabel('predicted'); ax.set_ylabel('actual')
plt.tight_layout()
plt.show()

### Lab 5.1 -- Pick the Winner and Explain Why

In [ ]:
# Solution: Lab 5.1
# 1. Scorecard
results = {name: accuracy_score(y_test, preds) for name, preds in predictions.items()}

# 2. Winner
winner = max(results, key=results.get)
print(f'Winner: {winner}  ({results[winner]:.4f})')

# 3. Reasoning
reasoning = (
    'Logistic Regression typically wins on TF-IDF sentiment because TF-IDF features are '
    'roughly linearly separable for polarized text (words like "excellent" push the '
    'decision in one direction, "terrible" in the other). Multinomial NB is a close '
    'second and trains orders of magnitude faster. Random Forest struggles here because '
    'bagged trees on a ~20K-dim sparse matrix each see tiny random feature subsets and '
    'rarely find useful splits -- classical text is a linear-model sweet spot.'
)
print(f'\nWhy: {reasoning}')

# Note: the actual ranking can flip by a few tenths of a percent across runs/seeds.
# The *shape* of the ranking (LR >= NB > RF, usually) is the robust finding.

## Section 6: Sentiment Scores & Feature Engineering

TextBlob polarity is a **rule-based** score in `[-1, +1]`. Frame it as a *feature*, not a classifier. We stack it alongside review length, exclamation count, and uppercase ratio onto the TF-IDF matrix via `scipy.sparse.hstack`.

In [ ]:
def hand_features(texts):
    feats = np.zeros((len(texts), 4), dtype=np.float64)
    for i, t in enumerate(texts):
        words = t.split()
        feats[i, 0] = len(t)
        feats[i, 1] = t.count('!')
        feats[i, 2] = sum(w.isupper() for w in words) / max(len(words), 1)
        feats[i, 3] = TextBlob(t).sentiment.polarity
    return feats

print('Computing hand-crafted features... (~30-60s)')
train_hand = hand_features(X_train.tolist())
test_hand  = hand_features(X_test.tolist())
print('Shapes:', train_hand.shape, test_hand.shape)
print('\nFirst 3 rows (length, excl, upper_ratio, polarity):')
print(np.round(train_hand[:3], 3))

In [ ]:
polarity_train = train_hand[:, 3]
print('Mean polarity by class (train):')
for cls in [0, 1]:
    m = polarity_train[y_train.values == cls].mean()
    print(f'  class {cls} ({["negative","positive"][cls]}):  {m:+.3f}')

In [ ]:
X_train_combo = sp.hstack([X_train_tfidf, sp.csr_matrix(train_hand)]).tocsr()
X_test_combo  = sp.hstack([X_test_tfidf,  sp.csr_matrix(test_hand)]).tocsr()
print(f'Combined train shape: {X_train_combo.shape}')

lr_combo = LogisticRegression(max_iter=1000, random_state=SEED).fit(X_train_combo, y_train)
acc_combo = accuracy_score(y_test, lr_combo.predict(X_test_combo))
print(f'\nLogReg + TF-IDF only          : {accuracy_score(y_test, predictions["LogisticRegression"]):.4f}')
print(f'LogReg + TF-IDF + hand-crafted: {acc_combo:.4f}')

### Lab 6.1 -- Engineer Two More Features

In [ ]:
# Solution: Lab 6.1
def extra_features(texts):
    feats = np.zeros((len(texts), 2), dtype=np.float64)
    for i, t in enumerate(texts):
        words = t.split()
        # Feature A: average word length (longer words ~ more thoughtful text)
        feats[i, 0] = np.mean([len(w) for w in words]) if words else 0.0
        # Feature B: TextBlob subjectivity [0, 1] -- 0=factual, 1=opinionated
        feats[i, 1] = TextBlob(t).sentiment.subjectivity
    return feats

train_extra = extra_features(X_train.tolist())
test_extra  = extra_features(X_test.tolist())

X_train_full = sp.hstack([X_train_combo, sp.csr_matrix(train_extra)]).tocsr()
X_test_full  = sp.hstack([X_test_combo,  sp.csr_matrix(test_extra)]).tocsr()

lr_full = LogisticRegression(max_iter=1000, random_state=SEED).fit(X_train_full, y_train)
acc_full = accuracy_score(y_test, lr_full.predict(X_test_full))

print(f'LogReg + TF-IDF + 4 hand + 2 extra: {acc_full:.4f}')
print(f'Delta vs 4-hand version           : {acc_full - acc_combo:+.4f}')

# Explanation:
# The lift from 2 more features is usually tiny (a few hundredths of a percent) because
# TF-IDF with bigrams already captures most signal. Hand-crafted features matter more on
# SHORT, noisy text where vocabulary is thin (e.g., tweets, SMS). A non-result is still a
# result: knowing which knobs *do not* move the needle saves time in production.

## Section 7: Final Lab -- SMS Spam Detection

Apply the full pipeline to the SMS Spam Collection (~5,500 English SMS labeled `ham`/`spam`). Deliverables: baseline, three classifiers, winner by macro-F1, confusion matrix.

In [ ]:
# Solution: Final Lab -- SMS Spam
SMS_URL = 'https://www.dropbox.com/scl/fi/yy0b8tblxx787vw0ncbm4/SMSSpamCollection.tsv?rlkey=iuk84q9leb2hcyisuvqe4c1hn&dl=1'

# 1. Load (TSV, no header)
sms = pd.read_csv(SMS_URL, sep='\t', names=['label', 'message'])
print(f'SMS dataset: {len(sms):,} rows')
print(sms['label'].value_counts())

# 2. Map labels + split
X_sms = sms['message']
y_sms = sms['label'].map({'ham': 0, 'spam': 1})
Xtr_sms, Xte_sms, ytr_sms, yte_sms = train_test_split(
    X_sms, y_sms, test_size=TEST_SIZE, random_state=SEED, stratify=y_sms)

# 3. Baseline
base_sms = accuracy_score(yte_sms, np.full(len(yte_sms), ytr_sms.mode()[0]))
print(f'\nSMS majority-class baseline: {base_sms:.4f}')

# 4. Vectorize
sms_vect = TfidfVectorizer(ngram_range=(1, 2), min_df=2, stop_words='english')
Xtr_vec = sms_vect.fit_transform(Xtr_sms)
Xte_vec = sms_vect.transform(Xte_sms)
print(f'SMS TF-IDF shape: {Xtr_vec.shape}')

# 5. Train three models
sms_models = {
    'MultinomialNB'      : MultinomialNB(),
    'RandomForest'       : RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
    'LogisticRegression' : LogisticRegression(max_iter=1000, random_state=SEED),
}
sms_preds = {}
sms_scores = {}
for name, clf in sms_models.items():
    clf.fit(Xtr_vec, ytr_sms)
    p = clf.predict(Xte_vec)
    sms_preds[name] = p
    sms_scores[name] = f1_score(yte_sms, p, average='macro')

# 6. Classification reports + winner
for name, p in sms_preds.items():
    print(f'\n=== {name}  (macro-F1 = {sms_scores[name]:.4f}) ===')
    print(classification_report(yte_sms, p, target_names=['ham', 'spam']))

sms_winner = max(sms_scores, key=sms_scores.get)
print(f'\nWinner by macro-F1: {sms_winner}  ({sms_scores[sms_winner]:.4f})')

# 7. Confusion matrix for winner
cm = confusion_matrix(yte_sms, sms_preds[sms_winner])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['ham', 'spam'], yticklabels=['ham', 'spam'])
plt.title(f'SMS Spam -- {sms_winner}')
plt.xlabel('predicted'); plt.ylabel('actual')
plt.tight_layout()
plt.show()

# Explanation:
# MNB often wins on SMS spam because (a) the vocabulary is tiny and discrete, (b) spam
# messages contain a handful of extremely predictive words (FREE, WIN, txt, claim),
# and (c) word-independence is a perfectly fine assumption for short texts.
#
# Business framing: on a production spam filter, a false POSITIVE (tagging ham as spam)
# is worse than a false NEGATIVE (a spam text sneaks through) -- losing a real message
# from a friend is more costly than seeing one junk SMS. So in production we would tune
# the decision threshold to push precision on the 'spam' class well above 0.98.

## Wrap-up & Next Steps

Full pipeline from blank notebook to shippable classifier:
1. Frame binary classification + stratified split.
2. Vectorize (BoW / TF-IDF, tune n-grams, stop words, `max_features`, `min_df`).
3. Beat the majority baseline with MNB / RF / LR.
4. Read precision / recall / F1 and pick the right metric.
5. Engineer hand-crafted features, `hstack` with `scipy.sparse`.
6. Apply the exact same pipeline to a new dataset in ~5 lines of change.

### Self-check quiz -- answers
1. **Unigrams.** 15-token tweets make bigrams extremely rare (each appears ~once), so they act as noise features.
2. **Its IDF becomes `log(1) + 1 = 1` (or `log(N/(N+1)) + 1 ~ 1` smoothed) -- a flat multiplier.** Then with `max_df=0.95` or stop-word filtering, it's dropped entirely. That is the whole point of TF-IDF: let the math quiet the everywhere-words.
3. **No.** 99% of one class means the majority baseline *already* gets 99%. Your model hitting 98% is actually *worse* than predicting the majority class. Always compute the baseline first.

### Up next: Notebook 3
Logistic regression deep-dive, precision-recall/ROC curves, Gradient Boosting & AdaBoost, and the pivot to unsupervised clustering (K-Means, DBSCAN, dendrograms).
